# Convert paired PROJECT_LVL data to CLEAR input

This reproducible conversion creates adjacent vulnerable/fixed records for CLEAR detection and evaluation.

## Setup

In [ ]:
import json
from pathlib import Path

ROOT = Path('/root/CLEAR_Causal_Context_based_Agentic_Reasoning_for_Vulnerability_Detection2')
SOURCE = ROOT / 'dataset/java-pair/java_test_paired_PROJECT_LVL.json'
OUTPUT = ROOT / 'dataset/java-pair/java_test_paired_CLEAR_PROJECT_LVL.json'
SOURCE, OUTPUT

## Steps

In [ ]:
def load_rows(path: Path) -> list[dict]:
    with path.open(encoding='utf-8') as handle:
        payload = json.load(handle)
    if isinstance(payload, dict):
        payload = payload.get('data')
    if not isinstance(payload, list) or not all(isinstance(row, dict) for row in payload):
        raise ValueError('Expected a JSON list of objects or an object containing a data list')
    return payload

def normalize_cwe(row: dict) -> str:
    value = row.get('cwe_id') or row.get('cwe')
    if not value and row.get('cwe_ids'):
        value = row['cwe_ids'][0]
    return str(value or '').strip()

def convert_to_clear(rows: list[dict]) -> list[dict]:
    clear_rows = []
    for pair_position, row in enumerate(rows):
        before = row.get('func_before')
        after = row.get('func_after')
        if not isinstance(before, str) or not before.strip():
            raise ValueError(f'Row {pair_position}: func_before is empty')
        if not isinstance(after, str) or not after.strip():
            raise ValueError(f'Row {pair_position}: func_after is empty')
        common = {
            'pair_index': row.get('index', pair_position),
            'commit_hash': row.get('commit_hash', ''),
            'file_name': row.get('file_name', ''),
            'cve_id': row.get('cve_id', ''),
            'cwe': normalize_cwe(row),
        }
        clear_rows.append({'index': 2 * pair_position, 'func': before, 'target': 1, **common})
        clear_rows.append({'index': 2 * pair_position + 1, 'func': after, 'target': 0, **common})
    return clear_rows

In [ ]:
source_rows = load_rows(SOURCE)
clear_rows = convert_to_clear(source_rows)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT.open('w', encoding='utf-8') as handle:
    json.dump(clear_rows, handle, ensure_ascii=False, indent=2)

## Checks

In [ ]:
assert len(clear_rows) == 2 * len(source_rows)
assert len({row['index'] for row in clear_rows}) == len(clear_rows)
assert all(clear_rows[i]['target'] == 1 and clear_rows[i + 1]['target'] == 0
           for i in range(0, len(clear_rows), 2))
assert all(row['func'].strip() and row['cwe'] for row in clear_rows)
summary = {
    'source_pairs': len(source_rows),
    'clear_records': len(clear_rows),
    'vulnerable_records': sum(row['target'] == 1 for row in clear_rows),
    'fixed_records': sum(row['target'] == 0 for row in clear_rows),
    'output': str(OUTPUT),
}
summary

## Next Steps

Run `./run_project_lvl_test.sh` from the repository root.